# Notebook 5 — Structured Streaming

**Project phase:** F1 Race Strategy Intelligence Platform  
**Spark components:** Structured Streaming · `withWatermark` · Window Aggregations · `foreachBatch`  
**Goal:** Simulate a live race-engineer dashboard that tracks lap-by-lap position changes and gap-to-leader in real time.

## Architecture

```
lap_times.csv (sorted by raceId, lap)
        ↓
Python producer script
(copies per-lap CSV slices into /tmp/f1_stream_input/ every 2 seconds)
        ↓
Spark readStream (file source, maxFilesPerTrigger=1)
        ↓
withWatermark("lap_timestamp", "10 seconds")
        ↓
window("lap_timestamp", "90 seconds") + groupBy(driverId)
        ↓
Compute: position, gap_to_leader, stint_lap, pit_flag
        ↓
writeStream (outputMode="update", foreachBatch → console)
writeStream (outputMode="append", Parquet audit log)
```

## Streaming features demonstrated

| Feature | Implementation |
|---|---|
| File source with controlled ingestion | `readStream.option("maxFilesPerTrigger", 1)` |
| Late-data handling | `withWatermark("lap_timestamp", "10 seconds")` |
| Windowed aggregation | `groupBy(window(...), "driverId").agg(...)` |
| Output modes | `outputMode("update")` for leaderboard; `outputMode("append")` for audit log |
| Fault tolerance | `option("checkpointLocation", ...)` on every query |
| Flexible micro-batch processing | `foreachBatch` with full DataFrame API |

## Business narrative

> *"This is what a race engineer sees in real time. As each lap completes, the system updates the leaderboard, flags drivers entering their pit window based on stint length, and highlights who is gaining or losing ground — all processed at scale with Spark Structured Streaming."*


## Google Colab Setup

Run this cell first. It installs the required libraries and mounts Google Drive. Raw CSV files must be available at `DATASET_PATH`.


In [8]:
!pip install -q pyspark pandas numpy pyarrow

from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/data_analysis"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports & SparkSession

Structured Streaming works with any standard SparkSession. We reduce shuffle partitions to 4 for Colab's single-node environment.


In [9]:
import os
import time
import threading

import pandas as pd
from datetime import datetime, timedelta

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("F1 Notebook 5 - Structured Streaming")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.streaming.schemaInference", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
spark


Spark version: 4.0.2


## Directory Setup

The streaming pipeline needs four directories:

- **Input dir** — watched by Spark `readStream`; producer drops lap files here
- **Archive dir** — producer moves processed files here after emission (optional housekeeping)
- **Checkpoint dir** — Spark stores offset progress and aggregation state (required for fault tolerance)
- **Audit log path** — Parquet sink for the append-mode audit log


In [10]:
STREAM_INPUT_DIR   = "/tmp/f1_stream_input"     # Spark watches this directory
STREAM_ARCHIVE_DIR = "/tmp/f1_stream_archive"    # optional housekeeping archive
CHECKPOINT_DIR     = "/tmp/f1_checkpoint_main"   # leaderboard query state
AUDIT_CHECKPOINT   = "/tmp/f1_checkpoint_audit"  # audit log query state
AUDIT_LOG_PATH     = "/tmp/f1_audit_log"         # Parquet output

for d in [STREAM_INPUT_DIR, STREAM_ARCHIVE_DIR,
           CHECKPOINT_DIR, AUDIT_CHECKPOINT, AUDIT_LOG_PATH]:
    os.makedirs(d, exist_ok=True)

print("Directories ready.")

# BIG-DATA NOTE: A checkpoint directory is mandatory for stateful streaming.
# It stores:
#   - Source offsets (which files/Kafka offsets have been processed)
#   - Aggregation state (window counts, sums etc. across micro-batches)
# Without it, a query restart reprocesses all data and re-initialises all state.
# Each concurrent streaming query must have its OWN checkpoint directory.


Directories ready.


## Explicit Schema for the Stream Source

`inferSchema=True` is not supported for streaming file sources — Spark cannot scan ahead on an unbounded stream. We define the schema explicitly including the synthetic `lap_timestamp` event-time column.


In [11]:
# BIG-DATA NOTE: inferSchema=False is mandatory for all streaming sources.
# Schema inference requires a full pass over the data, which is impossible
# for an unbounded stream.

lap_stream_schema = T.StructType([
    T.StructField("raceId",        T.IntegerType(),   True),
    T.StructField("driverId",      T.IntegerType(),   True),
    T.StructField("lap",           T.IntegerType(),   True),
    T.StructField("position",      T.IntegerType(),   True),
    T.StructField("time",          T.StringType(),    True),  # e.g. "1:32.456"
    T.StructField("milliseconds",  T.LongType(),      True),  # lap time in ms
    T.StructField("lap_timestamp", T.TimestampType(), True),  # synthetic event-time
])



## Target Race Configuration

Set `TARGET_RACE_ID` to any `raceId` present in the dataset. The default is Bahrain 2024 (raceId 1096). The producer will stream all laps for this race.


In [15]:
TARGET_RACE_ID = 1096   # Bahrain Grand Prix 2024
                        # Change to any raceId in races.csv

# Verify the race exists in the dataset
races_schema = T.StructType([
    T.StructField("raceId",    T.IntegerType(), True),
    T.StructField("year",      T.IntegerType(), True),
    T.StructField("round",     T.IntegerType(), True),
    T.StructField("circuitId", T.IntegerType(), True),
    T.StructField("name",      T.StringType(),  True),
    T.StructField("date",      T.DateType(),    True),
    T.StructField("time",      T.StringType(),  True),
    T.StructField("url",       T.StringType(),  True),
    T.StructField("fp1_date",  T.DateType(),    True), T.StructField("fp1_time",   T.StringType(), True),
    T.StructField("fp2_date",  T.DateType(),    True), T.StructField("fp2_time",   T.StringType(), True),
    T.StructField("fp3_date",  T.DateType(),    True), T.StructField("fp3_time",   T.StringType(), True),
    T.StructField("quali_date",T.DateType(),    True), T.StructField("quali_time", T.StringType(), True),
    T.StructField("sprint_date",T.DateType(),   True), T.StructField("sprint_time",T.StringType(), True),
])

race_info = (
    spark.read
    .options(header="true", nullValue="\\N")
    .schema(races_schema)
    .csv(f"races.csv")
    .filter(F.col("raceId") == TARGET_RACE_ID)
    .select("raceId", "year", "name", "date")
)

race_info.show(truncate=False)


+------+----+--------------------+----------+
|raceId|year|name                |date      |
+------+----+--------------------+----------+
|1096  |2022|Abu Dhabi Grand Prix|2022-11-20|
+------+----+--------------------+----------+



## Load Static Dimension Tables

The driver lookup and grid position tables are loaded as standard batch DataFrames. We `broadcast` them for use inside the streaming join — Spark natively supports stream-to-batch joins.


In [19]:
driver_schema = T.StructType([
    T.StructField("driverId",    T.IntegerType(), True),
    T.StructField("driverRef",   T.StringType(),  True),
    T.StructField("number",      T.IntegerType(), True),
    T.StructField("code",        T.StringType(),  True),
    T.StructField("forename",    T.StringType(),  True),
    T.StructField("surname",     T.StringType(),  True),
    T.StructField("dob",         T.DateType(),    True),
    T.StructField("nationality", T.StringType(),  True),
    T.StructField("url",         T.StringType(),  True),
])

results_schema = T.StructType([
    T.StructField("resultId",        T.IntegerType(), True),
    T.StructField("raceId",          T.IntegerType(), True),
    T.StructField("driverId",        T.IntegerType(), True),
    T.StructField("constructorId",   T.IntegerType(), True),
    T.StructField("number",          T.IntegerType(), True),
    T.StructField("grid",            T.IntegerType(), True),
    T.StructField("position",        T.IntegerType(), True),
    T.StructField("positionText",    T.StringType(),  True),
    T.StructField("positionOrder",   T.IntegerType(), True),
    T.StructField("points",          T.DoubleType(),  True),
    T.StructField("laps",            T.IntegerType(), True),
    T.StructField("time",            T.StringType(),  True),
    T.StructField("milliseconds",    T.LongType(),    True),
    T.StructField("fastestLap",      T.IntegerType(), True),
    T.StructField("rank",            T.IntegerType(), True),
    T.StructField("fastestLapTime",  T.StringType(),  True),
    T.StructField("fastestLapSpeed", T.StringType(),  True),
    T.StructField("statusId",        T.IntegerType(), True),
])

# BIG-DATA NOTE: broadcast() is safe here because drivers.csv is < 1 MB.
# Spark replicates the broadcasted table to all executors, avoiding a shuffle join.
dim_drivers = (
    spark.read
    .options(header="true", nullValue="\\N")
    .schema(driver_schema)
    .csv(f"drivers.csv")
    .select(
        "driverId",
        F.concat_ws(" ", "forename", "surname").alias("driver_name"),
        "code",
        "nationality",
    )
)

dim_grid = (
    spark.read
    .options(header="true", nullValue="\\N")
    .schema(results_schema)
    .csv(f"results.csv")
    .filter(F.col("raceId") == TARGET_RACE_ID)
    .select("driverId", F.col("grid").alias("grid_position"))
)

print(f"dim_drivers: {dim_drivers.count()} rows")
print(f"dim_grid (race {TARGET_RACE_ID}): {dim_grid.count()} rows")
dim_grid.orderBy("grid_position").show(20, truncate=False)


dim_drivers: 861 rows
dim_grid (race 1096): 20 rows
+--------+-------------+
|driverId|grid_position|
+--------+-------------+
|830     |1            |
|815     |2            |
|844     |3            |
|832     |4            |
|1       |5            |
|847     |6            |
|846     |7            |
|839     |8            |
|20      |9            |
|4       |10           |
|852     |11           |
|854     |12           |
|817     |13           |
|840     |14           |
|855     |15           |
|825     |16           |
|842     |17           |
|822     |18           |
|848     |19           |
|849     |20           |
+--------+-------------+



## Producer — Simulating Live Race Telemetry

In a real F1 deployment the stream source would be Apache Kafka carrying live car telemetry at sub-second intervals. Here we simulate that by reading a historical race from `lap_times.csv`, enriching each lap with a synthetic timestamp, and writing one lap's data as a CSV file into the watched directory every 2 seconds.

The producer runs in a **background daemon thread** so it does not block the main thread where the Spark streaming query runs.


In [20]:
def prepare_race_laps(race_id: int, lap_times_path: str) -> pd.DataFrame:
    """
    Load lap_times.csv for a single race, assign synthetic event-time timestamps
    (T0 + lap_number * 90 seconds), and sort chronologically.

    The synthetic timestamp is required because the raw lap_times.csv has no
    wall-clock timestamp — only lap numbers. Spark's watermark and window
    aggregations require a genuine TimestampType column to anchor event-time.
    """
    laps = pd.read_csv(
        lap_times_path,
        dtype={"raceId": int, "driverId": int, "lap": int,
               "position": int, "time": str, "milliseconds": "Int64"},
        na_values=["\\N", ""],
    )

    race_laps = laps[laps["raceId"] == race_id].copy()
    if race_laps.empty:
        raise ValueError(f"No lap data for raceId={race_id}. "
                         f"Available IDs: {sorted(laps['raceId'].unique())[:10]}")

    # Synthetic event-time: race starts at T0, each lap adds ~90 seconds
    T0 = datetime(2024, 3, 2, 14, 0, 0)   # Bahrain 2024 start time
    race_laps["lap_timestamp"] = race_laps["lap"].apply(
        lambda n: (T0 + timedelta(seconds=n * 90)).strftime("%Y-%m-%d %H:%M:%S")
    )

    return race_laps.sort_values(["lap", "driverId"]).reset_index(drop=True)


def stream_producer(race_id: int, lap_times_path: str,
                    output_dir: str, interval_secs: float = 2.0,
                    max_laps: int = None, verbose: bool = True):
    """
    Background producer that drops one lap-worth of data as a CSV file
    into output_dir every interval_secs seconds, simulating live telemetry.

    Parameters
    ----------
    race_id        : raceId to stream from lap_times.csv
    lap_times_path : path to the raw lap_times.csv file
    output_dir     : directory watched by Spark readStream
    interval_secs  : seconds between lap file drops (default 2)
    max_laps       : stop after N laps (None = full race)
    verbose        : print each lap drop
    """
    laps_df = prepare_race_laps(race_id, lap_times_path)
    lap_numbers = sorted(laps_df["lap"].unique())
    if max_laps:
        lap_numbers = lap_numbers[:max_laps]

    for lap_num in lap_numbers:
        lap_slice = laps_df[laps_df["lap"] == lap_num]
        fname = os.path.join(output_dir, f"lap_{lap_num:03d}.csv")
        lap_slice.to_csv(fname, index=False)
        if verbose:
            n_drivers = len(lap_slice)
            print(f"[Producer] Lap {lap_num:3d} → {fname}  ({n_drivers} drivers)")
        time.sleep(interval_secs)

    print("[Producer] All laps emitted.")


print("Producer functions defined.")


Producer functions defined.


## `readStream` — File Source

`maxFilesPerTrigger=1` is the key backpressure knob: Spark processes exactly one new file (one lap) per micro-batch. This mirrors the cadence of real-time lap updates and prevents a processing backlog during burst scenarios.


In [21]:
# BIG-DATA NOTE: maxFilesPerTrigger=1 is our backpressure control.
# In production with Kafka the equivalent is `maxOffsetsPerTrigger`.
# Without it, a burst of 57 lap files arriving at once would be processed
# in a single huge batch, inflating latency for the first output.

raw_stream = (
    spark.readStream
    .format("csv")
    .schema(lap_stream_schema)              # explicit schema — no inferSchema
    .option("header", "true")
    .option("maxFilesPerTrigger", 1)        # one lap file per micro-batch
    .option("nullValue", "\\N")
    .option("latestFirst", "false")         # process laps in chronological order
    .load(STREAM_INPUT_DIR)
)

print("Streaming DataFrame schema:")
raw_stream.printSchema()
print(f"isStreaming: {raw_stream.isStreaming}")


Streaming DataFrame schema:
root
 |-- raceId: integer (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- lap: integer (nullable = true)
 |-- position: integer (nullable = true)
 |-- time: string (nullable = true)
 |-- milliseconds: long (nullable = true)
 |-- lap_timestamp: timestamp (nullable = true)

isStreaming: True


## Watermark — Late-Data Handling

The watermark declares how late arriving data can be while still being included in a window. We allow 10 seconds:

- If a lap file arrives ≤ 10 s late it is included in the correct lap window.
- Once the watermark advances past a window's end time, that window is **finalised** and its state is dropped from memory.

This is critical at scale: without a watermark, Spark would retain every window's state forever, causing an OOM failure over a long race season.


In [22]:
# BIG-DATA NOTE: withWatermark is required for:
#   1. Window aggregations on a stream
#   2. Stream deduplication
#   3. Stream-stream joins
# It bounds the amount of state Spark holds in memory by defining
# when old windows can be safely evicted.

watermarked_stream = (
    raw_stream
    .withWatermark("lap_timestamp", "10 seconds")
)

print("Watermark applied: 10-second late-data tolerance on 'lap_timestamp'")


Watermark applied: 10-second late-data tolerance on 'lap_timestamp'


## Windowed Aggregation — Leaderboard per Lap

We group by a 90-second tumbling window (matching our synthetic lap duration) and `driverId` to produce one leaderboard row per driver per lap. The output is enriched with driver name and grid position via broadcast joins.


In [32]:
leaderboard_stream = (
    watermarked_stream
    .groupBy(
        F.window("lap_timestamp", "90 seconds"),
        "driverId"
    )
    .agg(
        F.min("lap").alias("lap_number"),
        F.min("position").alias("current_position"),
        F.min("milliseconds").alias("lap_time_ms"),
    )
    .join(F.broadcast(dim_drivers), on="driverId", how="left")
    .join(F.broadcast(dim_grid),    on="driverId", how="left")
    .withColumn("position_change",
        F.col("grid_position") - F.col("current_position"))
    .withColumn("lap_time_display",
        F.concat_ws(
            ":",
            (F.col("lap_time_ms") / 60000).cast(T.IntegerType()).cast(T.StringType()),
            F.lpad(
                ((F.col("lap_time_ms") % 60000) / 1000)
                .cast(T.DecimalType(6, 3)).cast(T.StringType()),
                6, "0"
            )
        )
    )
    .select(
        F.col("window.start").alias("window_start"),
        "driverId",          # ← keep this so pit alerts can join on it
        "lap_number",
        "current_position",
        "driver_name",
        "code",
        "lap_time_display",
        "lap_time_ms",
        "position_change",
        "grid_position",
    )
)

print("Leaderboard streaming DataFrame defined.")
leaderboard_stream.printSchema()

Leaderboard streaming DataFrame defined.
root
 |-- window_start: timestamp (nullable = true)
 |-- driverId: integer (nullable = true)
 |-- lap_number: integer (nullable = true)
 |-- current_position: integer (nullable = true)
 |-- driver_name: string (nullable = true)
 |-- code: string (nullable = true)
 |-- lap_time_display: string (nullable = false)
 |-- lap_time_ms: long (nullable = true)
 |-- position_change: integer (nullable = true)
 |-- grid_position: integer (nullable = true)



## `foreachBatch` Handlers

`foreachBatch` is the most flexible Spark Structured Streaming sink. For each micro-batch trigger it delivers a **static** batch DataFrame containing only the rows from that trigger. We can then apply any Spark operation — window functions, additional joins, or Pandas conversion for display.

We implement two handlers:
1. **`compute_leaderboard`** — renders the gap-to-leader timing screen
2. **`compute_pit_alerts`** — flags drivers entering their pit window


In [33]:
pit_stops_schema = T.StructType([
    T.StructField("raceId",       T.IntegerType(), True),
    T.StructField("driverId",     T.IntegerType(), True),
    T.StructField("stop",         T.IntegerType(), True),
    T.StructField("lap",          T.IntegerType(), True),
    T.StructField("time",         T.StringType(),  True),
    T.StructField("duration",     T.StringType(),  True),
    T.StructField("milliseconds", T.LongType(),    True),
])

# Load pit stop data once; re-used inside foreachBatch to find last pit lap
pit_stops_df = (
    spark.read
    .options(header="true", nullValue="\\N")
    .schema(pit_stops_schema)
    .csv(f"pit_stops.csv")
    .filter(F.col("raceId") == TARGET_RACE_ID)
    .select("driverId", "lap", "stop")
    .cache()   # cache because this is read inside every micro-batch
)

print(f"Pit stops loaded: {pit_stops_df.count()} rows (cached)")


def compute_leaderboard(batch_df, batch_id):
    """
    Compute gap-to-leader and render the timing screen for one micro-batch.
    The batch_df is a static DataFrame — we can use any Spark operation here,
    including window functions that are not available directly on streaming DFs.
    """
    if batch_df.rdd.isEmpty():
        return

    # Cumulative race time per driver up to this lap
    # (batch_df already contains only this lap's rows for all drivers)
    # We use a window spec over lap_number to accumulate across batches
    # by including cumulative_ms in the batch schema (see note below)

    # Gap-to-leader within this single batch (one lap)
    best_time_this_lap = batch_df.agg(F.min("lap_time_ms")).collect()[0][0]

    enriched = (
        batch_df
        .withColumn("gap_ms",
            F.col("lap_time_ms") - F.lit(best_time_this_lap))
        .withColumn("gap_display",
            F.when(F.col("current_position") == 1, F.lit("LEADER"))
             .otherwise(
                 F.concat(
                     F.lit("+"),
                     (F.col("gap_ms") / 1000).cast(T.DecimalType(7, 3)).cast(T.StringType()),
                     F.lit("s")
                 )
             )
        )
        .withColumn("position_change_label",
            F.when(F.col("position_change") > 0,
                   F.concat(F.lit("\u25b2"), F.col("position_change").cast(T.StringType())))
             .when(F.col("position_change") < 0,
                   F.concat(F.lit("\u25bc"), F.abs("position_change").cast(T.StringType())))
             .otherwise(F.lit("\u2014"))
        )
    )

    # BIG-DATA NOTE: .collect() is used here AFTER all aggregations are complete.
    # We collect at most 20 driver rows — a bounded, predictable result set.
    # Never call .collect() before aggregation on the full dataset.
    rows = (
        enriched
        .select("current_position", "code", "driver_name",
                "lap_number", "lap_time_display", "gap_display",
                "position_change_label")
        .orderBy("current_position")
        .limit(20)
        .collect()
    )

    lap_num = rows[0].lap_number if rows else "?"
    print(f"\n{'='*72}")
    print(f" LIVE LEADERBOARD  |  Batch {batch_id}  |  Lap {lap_num}")
    print(f"{'='*72}")
    hdr = f"{'P':>3} {'COD':>4}  {'DRIVER':<22} {'LAP':>4} {'LAP TIME':>9}  {'GAP':>12}  {'\u0394POS':>6}"
    print(hdr)
    print("-" * 72)
    for r in rows:
        print(
            f"{r.current_position:>3} "
            f"{str(r.code or '???'):>4}  "
            f"{str(r.driver_name or 'Unknown'):<22} "
            f"{str(r.lap_number or '-'):>4} "
            f"{str(r.lap_time_display or '-'):>9}  "
            f"{str(r.gap_display or '-'):>12}  "
            f"{str(r.position_change_label or '-'):>6}"
        )


def compute_pit_alerts(batch_df, batch_id):
    """
    Flag drivers whose stint length >= 15 laps (entering pit window).
    batch_df now retains driverId so the join to last_pit resolves correctly.
    """
    if batch_df.rdd.isEmpty():
        return

    current_lap = batch_df.agg(F.max("lap_number")).collect()[0][0]
    if current_lap is None:
        return

    last_pit = (
        pit_stops_df
        .filter(F.col("lap") <= current_lap)
        .groupBy("driverId")
        .agg(F.max("lap").alias("last_pit_lap"))
    )

    alerts = (
        batch_df
        .join(F.broadcast(last_pit), on="driverId", how="left")
        .withColumn("last_pit_lap", F.coalesce(F.col("last_pit_lap"), F.lit(0)))
        .withColumn("stint_lap", F.col("lap_number") - F.col("last_pit_lap"))
        .filter(F.col("stint_lap") >= 15)
        .orderBy("current_position")
        .select("current_position", "code", "driver_name", "stint_lap")
        .collect()
    )

    if alerts:
        print(f"\n⚑  PIT WINDOW ALERTS  |  Lap {current_lap}")
        print(f"{'P':>3} {'COD':>4}  {'DRIVER':<22} {'STINT LAP':>10}")
        print("-" * 46)
        for r in alerts:
            print(f"{r.current_position:>3} {str(r.code or '???'):>4}  "
                  f"{str(r.driver_name or 'Unknown'):<22} {r.stint_lap:>10}")

print("compute_pit_alerts() fixed.")

def process_batch(batch_df, batch_id):
    """Combined foreachBatch handler: leaderboard + pit alerts."""
    compute_leaderboard(batch_df, batch_id)
    compute_pit_alerts(batch_df, batch_id)


print("foreachBatch handlers defined.")


Pit stops loaded: 31 rows (cached)
compute_pit_alerts() fixed.
foreachBatch handlers defined.


## Audit Log Sink — `append` Mode to Parquet

A second streaming query writes every finalised lap row to a Parquet audit log partitioned by `lap_number`. This uses `outputMode("append")`: rows are written only once the watermark advances past the window's end time, guaranteeing immutability.

The audit log feeds the downstream Bronze layer of the medallion architecture and enables post-race analysis without re-running the stream.


In [ ]:

audit_query = (
    leaderboard_stream
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", AUDIT_LOG_PATH)
    .option("checkpointLocation", AUDIT_CHECKPOINT)
    .partitionBy("lap_number")           # efficient downstream reads by lap
    .trigger(processingTime="4 seconds")
    .start()
)

print(f"Audit log sink started → {AUDIT_LOG_PATH}")
print(f"Partitioned by: lap_number")
print(f"Output mode:    append (rows written once, after watermark advances)")


Audit log sink started → /tmp/f1_audit_log
Partitioned by: lap_number
Output mode:    append (rows written once, after watermark advances)


## Main Streaming Query — Live Leaderboard

This cell starts the leaderboard query and the lap producer in parallel. Adjust `MAX_LAPS_TO_STREAM` and `PRODUCER_INTERVAL` to control how much of the race is streamed and how quickly.

`outputMode("update")` emits rows as soon as any window receives new data, giving low-latency updates for the console display — no need to wait for the watermark to finalise the window.


In [34]:
# ── Configuration ───────────────────────────────────────────────────────────
MAX_LAPS_TO_STREAM = 20      # set to None for full race (57 laps for Bahrain 2024)
PRODUCER_INTERVAL  = 2.0    # seconds between lap file drops
STREAM_TIMEOUT     = 120    # seconds before auto-stop

# ── Start producer in a daemon background thread ─────────────────────────────
producer_thread = threading.Thread(
    target=stream_producer,
    kwargs=dict(
        race_id       = TARGET_RACE_ID,
        lap_times_path= f"lap_times.csv",
        output_dir    = STREAM_INPUT_DIR,
        interval_secs = PRODUCER_INTERVAL,
        max_laps      = MAX_LAPS_TO_STREAM,
        verbose       = True,
    ),
    daemon=True,
)
producer_thread.start()
print("[Main] Producer thread started.")

# Give the producer a head-start so the first file exists when Spark polls
time.sleep(3)

# ── Start main leaderboard query ─────────────────────────────────────────────
# BIG-DATA NOTE: outputMode('update') emits updated rows immediately.
# foreachBatch gives us a static batch DF per trigger — full Spark API available.

leaderboard_query = (
    leaderboard_stream
    .writeStream
    .outputMode("update")
    .foreachBatch(process_batch)
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime="4 seconds")   # micro-batch every 4 seconds
    .start()
)

print("[Main] Leaderboard query started.")
print(f"[Main] Will auto-stop after {STREAM_TIMEOUT}s or when producer finishes.\n")

try:
    producer_thread.join(timeout=STREAM_TIMEOUT)
    # Allow Spark to flush the final batch after the producer finishes
    time.sleep(PRODUCER_INTERVAL * 3)
finally:
    leaderboard_query.stop()
    audit_query.stop()
    print("[Main] All streaming queries stopped.")


[Main] Producer thread started.
[Producer] Lap   1 → /tmp/f1_stream_input/lap_001.csv  (20 drivers)
[Main] Leaderboard query started.
[Main] Will auto-stop after 120s or when producer finishes.

[Producer] Lap   2 → /tmp/f1_stream_input/lap_002.csv  (20 drivers)

 LIVE LEADERBOARD  |  Batch 0  |  Lap 1
  P  COD  DRIVER                  LAP  LAP TIME           GAP    ΔPOS
------------------------------------------------------------------------
  1  VER  Max Verstappen            1  1:32.198        LEADER       —
  2  PER  Sergio Pérez              1  1:33.251       +1.053s       —
  3  LEC  Charles Leclerc           1  1:34.192       +1.994s       —
  4  HAM  Lewis Hamilton            1  1:34.864       +2.666s      ▲1
  5  SAI  Carlos Sainz              1  1:35.436       +3.238s      ▼1
  6  NOR  Lando Norris              1  1:35.908       +3.710s      ▲1
  7  RUS  George Russell            1  1:36.506       +4.308s      ▼1
  8  OCO  Esteban Ocon              1  1:36.964       +4.766s  

## Stream Query Progress & Metrics

After the query stops, inspect the last progress report to understand micro-batch latency, throughput, and watermark state.


In [ ]:
import json

try:
    last_progress = leaderboard_query.lastProgress
    if last_progress:
        print("Last leaderboard query progress:")
        print(json.dumps(last_progress, indent=2, default=str))
    else:
        print("No progress recorded (query may not have processed any data yet).")
except Exception as e:
    print(f"Progress unavailable: {e}")


Last leaderboard query progress:
{
  "id": "e9f478e2-e0c8-4cae-8cec-5d6b9771610d",
  "runId": "0a9d35f6-dff0-4927-8def-92e3cbac8d6f",
  "name": null,
  "timestamp": "2026-05-31T17:47:25.758Z",
  "batchId": 9,
  "batchDuration": 4821,
  "durationMs": {
    "addBatch": 4557,
    "commitOffsets": 102,
    "getBatch": 9,
    "latestOffset": 62,
    "queryPlanning": 43,
    "triggerExecution": 4820,
    "walCommit": 43
  },
  "eventTime": {
    "avg": "2024-03-02T14:18:00.000Z",
    "max": "2024-03-02T14:18:00.000Z",
    "min": "2024-03-02T14:18:00.000Z",
    "watermark": "2024-03-02T14:16:20.000Z"
  },
  "stateOperators": [
    {
      "operatorName": "stateStoreSave",
      "numRowsTotal": 332,
      "numRowsUpdated": 108,
      "numRowsRemoved": 110,
      "allUpdatesTimeMs": 1084,
      "allRemovalsTimeMs": 31,
      "commitTimeMs": 1790,
      "memoryUsedBytes": 93120,
      "numRowsDroppedByWatermark": 0,
      "numShufflePartitions": 4,
      "numStateStoreInstances": 22,
      "cust

## Post-Stream: Read Back the Audit Log

After streaming completes, the Parquet audit log is available as a standard batch DataFrame. This demonstrates the full pipeline: real-time stream → immutable Parquet store → batch analytical query.


In [ ]:
try:
    audit_df = spark.read.parquet(AUDIT_LOG_PATH)

    total_rows = audit_df.count()
    distinct_laps = audit_df.select("lap_number").distinct().count()

    print(f"Audit log summary")
    print(f"  Total rows    : {total_rows}")
    print(f"  Laps recorded : {distinct_laps}")

    print("\nFastest lap per driver across streamed laps:")
    (
        audit_df
        .groupBy("driver_name", "code")
        .agg(
            F.min("lap_time_ms").alias("fastest_lap_ms"),
            F.count("*").alias("laps_recorded")
        )
        .orderBy("fastest_lap_ms")
        .show(20, truncate=False)
    )

    print("\nPosition changes across streamed laps (biggest movers):")
    (
        audit_df
        .groupBy("driver_name", "code")
        .agg(
            F.max("position_change").alias("max_positions_gained"),
            F.min("position_change").alias("max_positions_lost"),
        )
        .orderBy(F.col("max_positions_gained").desc())
        .show(20, truncate=False)
    )

except Exception as e:
    print(f"Audit log not yet populated: {e}")
    print("Run the streaming cell above to generate data.")


Audit log summary
  Total rows    : 160
  Laps recorded : 8

Fastest lap per driver across streamed laps:
+----------------+----+--------------+-------------+
|driver_name     |code|fastest_lap_ms|laps_recorded|
+----------------+----+--------------+-------------+
|Max Verstappen  |VER |89968         |8            |
|Sergio Pérez    |PER |90272         |8            |
|Lewis Hamilton  |HAM |90346         |8            |
|Charles Leclerc |LEC |90418         |8            |
|George Russell  |RUS |90631         |8            |
|Carlos Sainz    |SAI |90685         |8            |
|Lando Norris    |NOR |91188         |8            |
|Esteban Ocon    |OCO |91385         |8            |
|Sebastian Vettel|VET |91439         |8            |
|Fernando Alonso |ALO |91539         |8            |
|Yuki Tsunoda    |TSU |91604         |8            |
|Lance Stroll    |STR |91740         |8            |
|Daniel Ricciardo|RIC |91815         |8            |
|Guanyu Zhou     |ZHO |91937         |8       

## Summary — Streaming Features Demonstrated

| Feature | Where used | Business purpose |
|---|---|---|
| `readStream` with `maxFilesPerTrigger=1` | Main query | Controlled ingestion, backpressure |
| `withWatermark("lap_timestamp", "10 seconds")` | Watermark cell | Late-data tolerance; bounds state memory |
| `window("lap_timestamp", "90 seconds")` | Leaderboard aggregation | Per-lap tumbling window grouping |
| `outputMode("update")` | Leaderboard query | Low-latency live display |
| `outputMode("append")` | Audit log query | Immutable historical record |
| `foreachBatch` | `process_batch` | Full DataFrame API per micro-batch |
| `checkpointLocation` | Both queries | Fault tolerance, exactly-once semantics |
| `broadcast()` for dimension joins | `dim_drivers`, `dim_grid`, `last_pit` | Avoids shuffle on stream-batch joins |
| Parquet audit log partitioned by `lap_number` | Audit sink | Efficient downstream analytical reads |
| Separate checkpoint dirs per query | Config | Prevents offset corruption |
